
# Fit24: bi-exponential decay model

``Fit24`` fits a bi-exponential fluorescence decay model in Jordi format. Use it
when the data contain two lifetime components and anisotropy is not part of the
model.

The optimized parameter vector is:

``[tau1, gamma, tau2, A2, offset]``
   ``tau1`` and ``tau2`` are the two lifetimes, ``A2`` is the amplitude of the
   second component, ``gamma`` is the scattered/background fraction, and
   ``offset`` is a constant background term.

This example creates deterministic synthetic data, fits it with the reusable
``Fit24`` class, and prints the recovered parameters.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import tttrlib


def make_jordi_irf(n_channels=64, period=32.0):
    """Return a two-channel Jordi-format IRF and its time axis."""
    time_axis = np.linspace(0.0, period, n_channels * 2)
    irf = (
        np.exp(-0.5 * ((time_axis - 2.0) / 0.25) ** 2)
        + np.exp(-0.5 * ((time_axis - 18.0) / 0.25) ** 2)
    )
    return irf.astype(np.float64), time_axis


np.random.seed(0)

irf, time_axis = make_jordi_irf()
background = np.zeros_like(irf) + 0.2
dt = time_axis[1] - time_axis[0]
period = 32.0
convolution_stop = len(irf) // 2 - 1
corrections = np.array([period, 1.0, 0.1, 0.1, convolution_stop])

true_parameters = np.array([4.0, 0.01, 0.5, 0.9, 1.0])
_setup24 = tttrlib.setup_vector(
    'fit24', dt=dt, period=period, convolution_stop=int(convolution_stop))
fit24 = tttrlib.DecayFit2('fit24', _setup24, irf.tolist())

problem = tttrlib.DecayFitProblem(2, len(irf) // 2, dt)
problem.irf = tttrlib.VectorDouble(irf.tolist())
problem.background = tttrlib.VectorDouble(background.tolist())

probability_model = np.asarray(fit24.model_curve(true_parameters, problem))

data = np.random.poisson(probability_model * 500_000 / probability_model.sum())

problem.data = tttrlib.VectorDouble(np.asarray(data, dtype=float).tolist())

initial = [3.5, 0.02, 0.7, 0.5, 1.0]
# 0 = free, -1 = held. Everything is fitted here.
constraints = tttrlib.DecayFitConstraints(tttrlib.VectorInt32([0, 0, 0, 0, 0]))
outcome = fit24.fit(initial, constraints, problem)
result = {"x": np.asarray(outcome.parameters), "twoIstar": outcome.objective}

plt.plot(data, label="synthetic data")
plt.plot(np.asarray(problem.model), label="fit24 model")
plt.xlabel("microtime channel")
plt.ylabel("counts")
plt.legend()
plt.show()

print("Fit24 recovered parameters")
print("==========================")
print(f"tau1:   {result['x'][0]:.3f} ns")
print(f"gamma:  {result['x'][1]:.4f}")
print(f"tau2:   {result['x'][2]:.3f} ns")
print(f"A2:     {result['x'][3]:.3f}")
print(f"offset: {result['x'][4]:.3f}")
print(f"twoIstar: {result['twoIstar']:.3f}")